# Simple Team Allocation Example

10 employees — 3 ML engineers, 5 data scientists, 2 developers — assigned across 2 projects.

In [ ]:
from optimizer.models import (
    PersonInput, SkillLevel,
    ProjectInput, SkillRequirement,
    AssignmentWeights,
)
from optimizer.adapters.pulp_solver import PuLPTeamAssignmentSolver

## Employees

Skills are scored 0–5. Each role has a strong primary skill (4.5) and weak secondary ones.

In [3]:
ML  = 'machine_learning'
DS  = 'data_science'
DEV = 'software_development'

people = [
    # ── ML engineers ──────────────────────────────────────────────────────────
    PersonInput(id='ml_1', years_of_experience=6, skills=[SkillLevel(id=ML, level=4.5), SkillLevel(id=DS, level=2.0)]),
    PersonInput(id='ml_2', years_of_experience=4, skills=[SkillLevel(id=ML, level=4.0), SkillLevel(id=DS, level=1.5)]),
    PersonInput(id='ml_3', years_of_experience=8, skills=[SkillLevel(id=ML, level=5.0), SkillLevel(id=DS, level=3.0)]),

    # ── Data scientists ────────────────────────────────────────────────────────
    PersonInput(id='ds_1', years_of_experience=5, skills=[SkillLevel(id=DS, level=4.5), SkillLevel(id=ML, level=2.5)]),
    PersonInput(id='ds_2', years_of_experience=3, skills=[SkillLevel(id=DS, level=4.0), SkillLevel(id=ML, level=1.5)]),
    PersonInput(id='ds_3', years_of_experience=7, skills=[SkillLevel(id=DS, level=5.0), SkillLevel(id=ML, level=2.0)]),
    PersonInput(id='ds_4', years_of_experience=2, skills=[SkillLevel(id=DS, level=3.5), SkillLevel(id=ML, level=1.0)]),
    PersonInput(id='ds_5', years_of_experience=9, skills=[SkillLevel(id=DS, level=4.5), SkillLevel(id=ML, level=3.0)]),

    # ── Developers ─────────────────────────────────────────────────────────────
    PersonInput(id='dev_1', years_of_experience=5, skills=[SkillLevel(id=DEV, level=4.5)]),
    PersonInput(id='dev_2', years_of_experience=3, skills=[SkillLevel(id=DEV, level=4.0)]),
]

## Projects

| Project | Slots | Required roles |
|---------|-------|----------------|
| alpha   | 2     | 1 DS · 1 dev |
| beta    | 7     | 3 DS · 2 ML · 2 dev |

In [4]:
project_alpha = ProjectInput(
    id='alpha',
    n_slots=2,
    skill_requirements=[
        SkillRequirement(id=DS,  min_level=3.0),
        SkillRequirement(id=DEV, min_level=3.0),
    ],
)

project_beta = ProjectInput(
    id='beta',
    n_slots=7,
    skill_requirements=[
        SkillRequirement(id=DS,  min_level=3.0),
        SkillRequirement(id=ML,  min_level=3.0),
        SkillRequirement(id=DEV, min_level=3.0),
    ],
)

## Solve

In [5]:
weights = AssignmentWeights(performance=0.5, chemistry=0.1, growth=0.2, cost=0.2)
solver  = PuLPTeamAssignmentSolver()

result_alpha = solver.solve(project_alpha, people, weights)
result_beta  = solver.solve(project_beta,  people, weights)

AttributeError: 'SkillLevel' object has no attribute 'skill_id'

## Results

In [5]:
def show(result):
    print(f'Project {result.project_id}  (score={result.score})')
    for m in result.members:
        print(f'  {m.person_id:<8}  FTE={m.fte_allocation}')
    print()

show(result_alpha)
show(result_beta)

Project alpha  (score=0.85)
  ds_4      FTE=1.0
  ds_2      FTE=1.0

Project beta  (score=3.007778)
  ds_5      FTE=1.0
  ml_3      FTE=1.0
  ds_2      FTE=1.0
  ds_1      FTE=1.0
  ml_1      FTE=1.0
  ml_2      FTE=1.0
  ds_3      FTE=1.0

